In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
import random
import pickle
from collections import deque
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import seaborn as sns
from datetime import datetime
import json
from pathlib import Path
import warnings
import time
from tqdm import tqdm
warnings.filterwarnings('ignore')


# ============================================================================
# CONFIGURATION
# ============================================================================
INPUT_FOLDER   = "/home/user2/archive(9)/CIC_IOT_Dataset2023/MERGED_CSV"
OUTPUT_FOLDER  = "/home/user2/archive(9)"

# Paths to baseline files
BASELINE_DETECTOR_PATH       = "/home/user2/archive(9)/detector.pth"
BASELINE_SCALER_PATH         = "/home/user2/archive(9)/scaler.pkl"   # ← KEY FIX
BASELINE_RESULTS_PATH        = "/home/user2/archive(9)/results_summarybaseline.json"

USE_SAMPLE_SIZE      = 100000
LORA_FINETUNE_EPOCHS = 100      # ← was 30, now 100
RESPONSE_EPISODES    = 8000
BATCH_SIZE           = 256
LORA_LR              = 0.001    # ← was 0.0005, now matches baseline LR
SEED                 = 42

LORA_CONFIG = {
    'rank'       : 8,
    'alpha'      : 16,
    'dropout'    : 0.1,
    'use_lora'   : True,
    'lora_layers': 'all'
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================================
# LORA LINEAR LAYER
# ============================================================================
class LoRALinear(nn.Module):
    """
    LoRA-enhanced Linear layer.
    Output = xW + x(BA) * scaling
    Base W is FROZEN. Only lora_A and lora_B are trained.
    """
    def __init__(self, in_features, out_features, bias=True,
                 lora_rank=8, lora_alpha=16, lora_dropout=0.1, use_lora=True):
        super().__init__()
        self.use_lora = use_lora
        self.scaling  = lora_alpha / lora_rank

        self.linear = nn.Linear(in_features, out_features, bias=bias)

        if use_lora and lora_rank > 0:
            self.lora_A       = nn.Parameter(torch.zeros(lora_rank, in_features))
            nn.init.kaiming_uniform_(self.lora_A, a=np.sqrt(5))
            self.lora_B       = nn.Parameter(torch.zeros(out_features, lora_rank))
            nn.init.zeros_(self.lora_B)
            self.lora_dropout = nn.Dropout(lora_dropout)
        else:
            self.lora_A = self.lora_B = self.lora_dropout = None

    def forward(self, x):
        base = self.linear(x)
        if self.use_lora and self.lora_A is not None:
            lora = F.linear(self.lora_dropout(x),
                            self.lora_B @ self.lora_A) * self.scaling
            return base + lora
        return base

    def freeze_base(self):
        for p in self.linear.parameters():
            p.requires_grad = False

    def unfreeze_base(self):
        for p in self.linear.parameters():
            p.requires_grad = True


# ============================================================================
# LORA MODEL  (same architecture as baseline but with LoRALinear)
# ============================================================================
class LoRAThreatDetector(nn.Module):
    def __init__(self, input_dim=47, num_classes=12, lora_config=LORA_CONFIG):
        super().__init__()
        r  = lora_config['rank']
        a  = lora_config['alpha']
        do = lora_config['dropout']

        self.input_bn = nn.BatchNorm1d(input_dim)
        self.encoder  = nn.Sequential(
            LoRALinear(input_dim, 256, lora_rank=r, lora_alpha=a, lora_dropout=do),
            nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            LoRALinear(256, 256, lora_rank=r, lora_alpha=a, lora_dropout=do),
            nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
        )
        self.shortcut      = nn.Linear(input_dim, 256)   # no LoRA on shortcut
        self.deep_features = nn.Sequential(
            LoRALinear(256, 192, lora_rank=r, lora_alpha=a, lora_dropout=do),
            nn.BatchNorm1d(192), nn.ReLU(), nn.Dropout(0.25),
            LoRALinear(192, 128, lora_rank=r, lora_alpha=a, lora_dropout=do),
            nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
        )
        self.classifier = nn.Sequential(
            LoRALinear(128, 64, lora_rank=r, lora_alpha=a, lora_dropout=do),
            nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.15),
            LoRALinear(64, num_classes, lora_rank=r, lora_alpha=a, lora_dropout=do)
        )

    def forward(self, x):
        x        = self.input_bn(x)
        identity = self.shortcut(x)
        out      = F.relu(self.encoder(x) + identity)
        out      = self.deep_features(out)
        return self.classifier(out)

    def get_probabilities(self, x):
        return F.softmax(self.forward(x), dim=1)

    def get_lora_parameters(self):
        for name, param in self.named_parameters():
            if 'lora_' in name:
                yield param

    def count_lora_parameters(self):
        lora  = sum(p.numel() for p in self.get_lora_parameters())
        total = sum(p.numel() for p in self.parameters())
        return lora, total


# ============================================================================
# LOAD BASELINE WEIGHTS INTO LORA MODEL
# ============================================================================
def load_baseline_into_lora(lora_model, baseline_path, input_dim=47, num_classes=12):
    """
    Load detector.pth weights into LoRA model.
    Maps:  encoder.0.weight  →  encoder.0.linear.weight
    Then freezes encoder + deep_features base weights.
    Classifier head stays trainable (transfer learning best practice).
    LoRA adapters (lora_A, lora_B) always trainable.
    """
    print(f"\n📥 Loading baseline weights from: {baseline_path}")

    baseline_state = torch.load(baseline_path, map_location=device)
    lora_state     = lora_model.state_dict()
    new_state      = {}
    unmapped       = []

    for key, value in baseline_state.items():
        parts     = key.split('.')
        # Try inserting 'linear' before weight/bias for LoRALinear layers
        candidate = '.'.join(parts[:-1]) + '.linear.' + parts[-1]
        if candidate in lora_state:
            new_state[candidate] = value
        elif key in lora_state:
            new_state[key] = value          # direct match (BN, shortcut)
        else:
            unmapped.append(key)

    missing, unexpected = lora_model.load_state_dict(new_state, strict=False)
    lora_keys    = [k for k in missing if 'lora_' in k]
    other_missing = [k for k in missing if 'lora_' not in k]

    print(f"   ✓ Loaded      : {len(new_state)} weight tensors")
    print(f"   ✓ LoRA keys   : {len(lora_keys)} (initialised fresh — correct)")
    if other_missing:
        print(f"   ⚠ Still missing: {other_missing}")
    if unmapped:
        print(f"   ⚠ Unmapped keys: {unmapped}")

    # ── Freeze strategy (Solution B) ──────────────────────────────────────
    # encoder base weights    → FROZEN   (low-level features stay fixed)
    # shortcut base weights   → FROZEN
    # deep_features           → TRAINABLE (mid-level features adapt)
    # classifier head         → TRAINABLE (head fine-tuning)
    # lora_A / lora_B         → TRAINABLE (LoRA adapters always train)
    # BatchNorm / input_bn    → TRAINABLE (adapt to data distribution)
    for name, param in lora_model.named_parameters():
        if 'lora_' in name:
            param.requires_grad = True      # LoRA adapters always train
        elif 'classifier' in name:
            param.requires_grad = True      # classifier head trains
        elif 'deep_features' in name:
            param.requires_grad = True      # deep features also train
        elif 'input_bn' in name:
            param.requires_grad = True      # input BN adapts
        else:
            param.requires_grad = False     # encoder + shortcut frozen

    frozen    = sum(p.numel() for p in lora_model.parameters() if not p.requires_grad)
    trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
    lora_p, total_p = lora_model.count_lora_parameters()

    print(f"\n🔒 FREEZE STRATEGY")
    print(f"   Frozen params    : {frozen:,}")
    print(f"   Trainable params : {trainable:,}")
    print(f"   LoRA params      : {lora_p:,} / {total_p:,}")
    print(f"   Param efficiency : {(1 - lora_p/total_p)*100:.1f}%")

    return lora_model


# ============================================================================
# LOSS FUNCTION
# ============================================================================
class WeightedFocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, num_classes=12):
        super().__init__()
        if alpha is None:
            alpha = torch.ones(num_classes)
        self.register_buffer('alpha', alpha / alpha.sum())
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss    = F.cross_entropy(inputs, targets, reduction='none')
        pt         = torch.exp(-ce_loss)
        focal_loss = self.alpha[targets] * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()


# ============================================================================
# TRAIN LORA  (with test loss tracking)
# ============================================================================
def train_lora_only(model, X_train, y_train, X_test, y_test,
                    num_epochs=LORA_FINETUNE_EPOCHS, batch_size=BATCH_SIZE):
    """
    Fine-tune LoRA adapters + classifier head.
    Tracks both train loss AND test loss for the graph.
    """
    print("\n" + "=" * 70)
    print("LORA FINE-TUNING")
    print(f"  Epochs     : {num_epochs}")
    print(f"  LR         : {LORA_LR}")
    print(f"  Batch size : {batch_size}")
    print("=" * 70)

    # Confirm trainable params
    trainable_names = [n for n, p in model.named_parameters() if p.requires_grad]
    print(f"\n   Trainable param groups: {len(trainable_names)}")

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.15, random_state=SEED, stratify=y_train)

    X_val_tensor  = torch.FloatTensor(X_val).to(device)
    y_val_tensor  = torch.LongTensor(y_val).to(device)
    X_test_tensor = torch.FloatTensor(X_test).to(device)
    y_test_tensor = torch.LongTensor(y_test).to(device)

    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(
            torch.FloatTensor(X_tr), torch.LongTensor(y_tr)),
        batch_size=batch_size, shuffle=True, num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    criterion  = WeightedFocalLoss(gamma=2.0, num_classes=12).to(device)

    # Optimizer covers LoRA adapters + classifier head + BN
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(trainable_params, lr=LORA_LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=10)

    # ── Metric storage ────────────────────────────────────────────────────
    train_losses = []
    test_losses  = []       # ← NEW: test loss tracking
    train_accs   = []
    test_accs    = []

    best_val_acc = 0
    best_state   = None
    patience     = 35
    patience_ctr = 0

    for epoch in tqdm(range(num_epochs), desc="LoRA Fine-tuning"):
        # ── Train ─────────────────────────────────────────────────────────
        model.train()
        epoch_loss = epoch_correct = epoch_total = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            logits = model(X_batch)
            loss   = criterion(logits, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            optimizer.step()

            epoch_loss    += loss.item()
            epoch_correct += (logits.argmax(1) == y_batch).sum().item()
            epoch_total   += y_batch.size(0)

        avg_train_loss = epoch_loss / len(train_loader)
        train_acc      = 100 * epoch_correct / epoch_total

        # ── Validate + Test ───────────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            # Validation accuracy (for scheduler + early stop)
            val_logits = model(X_val_tensor)
            val_acc    = 100 * (val_logits.argmax(1) == y_val_tensor
                                ).float().mean().item()

            # Test accuracy
            test_logits    = model(X_test_tensor)
            test_acc       = 100 * (test_logits.argmax(1) == y_test_tensor
                                    ).float().mean().item()

            # Test loss  ← NEW
            avg_test_loss  = criterion(test_logits, y_test_tensor).item()

        train_losses.append(avg_train_loss)
        test_losses.append(avg_test_loss)       # ← store test loss
        train_accs.append(train_acc)
        test_accs.append(test_acc)

        scheduler.step(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch [{epoch+1}/{num_epochs}] "
                  f"Train Loss: {avg_train_loss:.4f} | "
                  f"Test Loss: {avg_test_loss:.4f} | "
                  f"Train: {train_acc:.2f}% | "
                  f"Val: {val_acc:.2f}% | "
                  f"Test: {test_acc:.2f}%")

        if patience_ctr >= patience:
            print(f"\n  Early stopping at epoch {epoch+1}")
            break

    if best_state:
        model.load_state_dict(best_state)
        print(f"\n✅ Best model restored (Val: {best_val_acc:.2f}%)")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        'train_losses'   : train_losses,
        'test_losses'    : test_losses,     # ← returned for graph
        'train_accs'     : train_accs,
        'test_accs'      : test_accs,
        'best_val_acc'   : best_val_acc,
        'final_test_acc' : test_accs[-1] if test_accs else 0
    }


# ============================================================================
# SAVE / LOAD LORA ADAPTER
# ============================================================================
def save_lora_adapter(model, save_path):
    lora_state = {name: param.data.cpu().clone()
                  for name, param in model.named_parameters()
                  if 'lora_' in name}
    torch.save(lora_state, save_path)
    size_mb = Path(save_path).stat().st_size / (1024 * 1024)
    print(f"\n💾 LoRA adapter saved: {save_path}  ({size_mb:.3f} MB)")
    return size_mb


def load_lora_adapter(model, load_path):
    lora_state = torch.load(load_path, map_location=device)
    for name, param in model.named_parameters():
        if name in lora_state:
            param.data = lora_state[name].to(param.device)
    print(f"📥 LoRA adapter loaded: {load_path}")
    return model


# ============================================================================
# DATA LOADER  — uses LOADED scaler, does NOT refit
# ============================================================================
class ImprovedDataLoader:
    def __init__(self, data_path=INPUT_FOLDER, sample_size=USE_SAMPLE_SIZE,
                 scaler=None):
        self.data_path   = Path(data_path) if data_path else None
        self.sample_size = sample_size
        # ── KEY FIX: use the scaler passed in, don't create a new one ──
        self.scaler      = scaler if scaler is not None else StandardScaler()

        self.classical_threats = {
            'Benign': 0, 'DDoS': 1, 'DoS': 2, 'Recon': 3,
            'Web_based': 4, 'BruteForce': 5, 'Spoofing': 6, 'Mirai': 7
        }
        self.pq_threats = {
            'PQ-Downgrade': 8, 'PQ-HNDL': 9, 'PQ-SideChannel': 10, 'PQ-Hybrid': 11
        }
        self.all_threats   = {**self.classical_threats, **self.pq_threats}
        self.threat_names  = list(self.all_threats.keys())
        self.num_classical = len(self.classical_threats)
        self.num_pq        = len(self.pq_threats)
        self.num_total     = len(self.all_threats)

    def load_dataset(self):
        X_classical, y_classical, num_features = self._load_classical_threats()
        pq_samples = int(len(X_classical) * 0.4)
        X_pq, y_pq = self._generate_distinct_pq_threats(
            X_classical, y_classical, num_features, pq_samples)

        X = np.vstack([X_classical, X_pq])
        y = np.concatenate([y_classical, y_pq])
        X, y = self._balance_classes(X, y)

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=SEED, stratify=y)

        print(f"\n✅ Dataset: {len(X_train):,} train | {len(X_test):,} test")
        print(f"   Features: {num_features}")
        return X_train, X_test, y_train, y_test, num_features

    def _balance_classes(self, X, y):
        unique, counts = np.unique(y, return_counts=True)
        target = int(np.mean(counts))
        Xb, yb = [], []
        for label in unique:
            mask = y == label
            Xc, yc = X[mask], y[mask]
            idx = np.random.choice(len(Xc), target, replace=len(Xc) < target)
            Xb.append(Xc[idx]); yb.append(yc[idx])
        X_ = np.vstack(Xb); y_ = np.concatenate(yb)
        s  = np.random.permutation(len(X_))
        return X_[s], y_[s]

    def _generate_distinct_pq_threats(self, X_cl, y_cl, nf, n_samples):
        X_pq, y_pq = [], []
        spc = n_samples // self.num_pq
        for pq_class in range(self.num_pq):
            for _ in range(spc):
                s = X_cl[np.random.randint(0, len(X_cl))].copy()
                if pq_class == 0:
                    s[np.random.choice(nf, nf//4,  replace=False)] *= np.random.uniform(0.3, 0.6)
                    s[np.random.choice(nf, nf//10, replace=False)] += np.random.uniform(1.5, 2.5)
                elif pq_class == 1:
                    idx = np.random.choice(nf, nf//3, replace=False)
                    s[idx] *= np.random.uniform(1.8, 2.5)
                elif pq_class == 2:
                    idx = np.random.choice(nf, nf//3, replace=False)
                    s[idx] *= np.random.choice([0.4, 2.0], len(idx))
                elif pq_class == 3:
                    s[np.random.choice(nf, nf//4, replace=False)] *= np.random.uniform(0.4, 0.7)
                    s[np.random.choice(nf, nf//4, replace=False)] *= np.random.uniform(1.6, 2.2)
                s += np.random.normal(0, 0.05, nf)
                s  = np.clip(s, -3, 4)
                X_pq.append(s); y_pq.append(self.num_classical + pq_class)
        return np.array(X_pq), np.array(y_pq)

    def _load_classical_threats(self):
        if not self.data_path or not self.data_path.exists():
            return self._generate_synthetic_classical_data()
        csv_files = sorted(self.data_path.glob("**/*.csv"))
        if not csv_files:
            return self._generate_synthetic_classical_data()
        dfs = []
        for f in csv_files[:5]:
            try: dfs.append(pd.read_csv(f, low_memory=False))
            except: pass
        if not dfs:
            return self._generate_synthetic_classical_data()
        df = pd.concat(dfs, ignore_index=True)
        if self.sample_size and len(df) > self.sample_size:
            df = df.sample(n=self.sample_size, random_state=SEED)
        label_col = next((c for c in df.columns if 'label' in c.lower()), None)
        if label_col is None:
            return self._generate_synthetic_classical_data()
        return self._preprocess_data(df, label_col)

    def _preprocess_data(self, df, label_col):
        y_raw = df[label_col]
        X     = df.drop(columns=[label_col])
        y     = self._map_labels(y_raw)
        X     = X.select_dtypes(include=[np.number])
        X     = X.replace([np.inf, -np.inf], np.nan).fillna(X.median(numeric_only=True))
        X     = X.loc[:, X.var(numeric_only=True) > 0]
        # ── KEY FIX: transform only, do NOT fit again ──
        Xs    = self.scaler.transform(X)
        return Xs, y, Xs.shape[1]

    def _map_labels(self, y_raw):
        m = np.zeros(len(y_raw), dtype=int)
        for i, l in enumerate(y_raw):
            ll = str(l).lower()
            if   'benign' in ll:              m[i] = 0
            elif 'ddos'   in ll:              m[i] = 1
            elif 'dos'    in ll:              m[i] = 2
            elif 'recon'  in ll or 'scan' in ll: m[i] = 3
            elif 'web'    in ll or 'sql'  in ll: m[i] = 4
            elif 'brute'  in ll:              m[i] = 5
            elif 'spoof'  in ll:              m[i] = 6
            elif 'mirai'  in ll or 'bot'  in ll: m[i] = 7
        return m

    def _generate_synthetic_classical_data(self):
        print("⚡ Generating synthetic classical data...")
        nf  = 47
        ns  = self.sample_size or 100000
        spc = ns // self.num_classical
        X, y = [], []
        patterns = [
            (0.2, 0.15, None,           None      ),
            (0.8, 0.2,  slice(0,  12),  (2.5, 3.5)),
            (0.75,0.18, slice(5,  17),  (2.2, 3.0)),
            (0.45,0.2,  slice(15, 27),  (1.7, 2.5)),
            (0.65,0.18, slice(20, 32),  (2.0, 2.8)),
            (0.78,0.17, slice(25, 37),  (2.3, 3.1)),
            (0.55,0.22, slice(30, 42),  (1.8, 2.6)),
            (0.88,0.15, slice(35, 47),  (2.6, 3.4)),
        ]
        for cid, (mu, sigma, slc, rng) in enumerate(patterns):
            for _ in range(spc):
                s = np.random.normal(mu, sigma, nf)
                if slc and rng:
                    s[slc] *= np.random.uniform(*rng)
                s += np.random.normal(0, 0.06, nf)
                s  = np.clip(s, -0.5, 2.0)
                X.append(s); y.append(cid)
        X = np.array(X); y = np.array(y)
        # ── KEY FIX: transform only if scaler already fitted ──
        try:
            Xs = self.scaler.transform(X)
        except Exception:
            Xs = self.scaler.fit_transform(X)
        return Xs, y, nf


# ============================================================================
# RESPONSE AGENT DQN
# ============================================================================
class ImprovedResponseDQN(nn.Module):
    def __init__(self, state_dim=15, action_dim=8):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(state_dim, 256), nn.LayerNorm(256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128),       nn.LayerNorm(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64),        nn.LayerNorm(64),  nn.ReLU(), nn.Dropout(0.15),
            nn.Linear(64, action_dim)
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        return self.network(x)


class BetterResponseEnvironment:
    def __init__(self, detector_model, threat_names, X_test, y_test):
        self.detector     = detector_model
        self.threat_names = threat_names
        self.X_test       = torch.FloatTensor(X_test).to(device)
        self.y_test       = y_test
        self.actions      = {
            0: "Ignore",         1: "Quick Scan",    2: "Full Scan",
            3: "Quarantine",     4: "Delete",        5: "Network Isolate",
            6: "PQ-Crypto Upgrade", 7: "Emergency Crypto Rotate"
        }
        self.optimal_actions = {
            0:0, 1:5, 2:2, 3:1, 4:4, 5:3, 6:2, 7:5, 8:6, 9:7, 10:6, 11:7
        }
        self.correct_action_reward = {
            0:10, 1:40, 2:30, 3:25, 4:35, 5:35, 6:30, 7:45,
            8:60, 9:70, 10:60, 11:70
        }
        self.reset()

    def reset(self):
        self.current_idx   = np.random.randint(0, len(self.X_test))
        sample             = self.X_test[self.current_idx:self.current_idx+1]
        self.true_label    = int(self.y_test[self.current_idx])
        with torch.no_grad():
            probs                = self.detector.get_probabilities(sample)
            self.detected_threat = int(torch.argmax(probs).item())
            self.confidence      = float(probs[0, self.detected_threat].item())
        self.system_health   = np.random.uniform(0.7, 1.0)
        self.crypto_strength = np.random.uniform(0.7, 1.0)
        return self._get_state()

    def _get_state(self):
        is_pq_true        = 1.0 if self.true_label     >= 8 else 0.0
        is_pq_detected    = 1.0 if self.detected_threat >= 8 else 0.0
        detection_correct = 1.0 if self.true_label == self.detected_threat else 0.0
        state = np.array([
            self.detected_threat / 11.0, self.confidence, detection_correct,
            is_pq_detected, is_pq_true, self.system_health, self.crypto_strength,
            (1 - self.confidence)*0.5 + (1 - detection_correct)*0.5,
            is_pq_true * (1 - self.crypto_strength),
            (1 - self.system_health) * 0.5,
            1.0 if self.detected_threat < 4           else 0.0,
            1.0 if 4 <= self.detected_threat < 8      else 0.0,
            1.0 if self.detected_threat >= 8           else 0.0,
            1.0 if self.confidence > 0.9 and self.detected_threat > 0 else 0.0,
            1.0 if is_pq_detected and self.crypto_strength < 0.8 else 0.0,
        ], dtype=np.float32)
        return torch.FloatTensor(state).unsqueeze(0).to(device)

    def step(self, action):
        optimal = self.optimal_actions[self.true_label]
        is_pq   = self.true_label >= 8
        if action == optimal:
            reward = self.correct_action_reward[self.true_label]
            if self.confidence > 0.9: reward += 15
            if is_pq:                 reward += 20
        else:
            if self.true_label == 0:
                reward = -40 if action in [3, 4, 5] else -20
            elif action == 0:
                reward = -80 if is_pq else -50
            else:
                reward = -40 if is_pq else -25
        if action == optimal:
            self.system_health = min(1.0, self.system_health + 0.15)
            if action in [6, 7]:
                self.crypto_strength = min(1.0, self.crypto_strength + 0.20)
        else:
            self.system_health = max(0.0, self.system_health - 0.25)
            if is_pq:
                self.crypto_strength = max(0.0, self.crypto_strength - 0.20)
        info = {
            'true_threat'    : self.threat_names[self.true_label],
            'detected_threat': self.threat_names[self.detected_threat],
            'confidence'     : self.confidence,
            'optimal_action' : self.actions[optimal],
            'taken_action'   : self.actions[action],
            'correct'        : action == optimal,
            'is_pq'          : bool(is_pq),
            'crypto_strength': self.crypto_strength
        }
        return self._get_state(), reward, True, info


class PrioritizedReplayBuffer:
    def __init__(self, capacity=30000):
        self.buffer     = deque(maxlen=capacity)
        self.priorities = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
        self.priorities.append(abs(reward) + 1.0)

    def sample(self, batch_size):
        probs   = np.array(self.priorities) / sum(self.priorities)
        indices = np.random.choice(len(self.buffer), batch_size, p=probs, replace=False)
        batch   = [self.buffer[i] for i in indices]
        s, a, r, ns, d = zip(*batch)
        return (torch.cat(s),
                torch.LongTensor(a).to(device),
                torch.FloatTensor(r).to(device),
                torch.cat(ns),
                torch.FloatTensor(d).to(device))

    def __len__(self):
        return len(self.buffer)


def train_response_agent(detector_model, threat_names, X_test, y_test,
                         num_episodes=RESPONSE_EPISODES, batch_size=256):
    print(f"\n🤖 TRAINING RESPONSE AGENT  (episodes: {num_episodes})")
    env        = BetterResponseEnvironment(detector_model, threat_names, X_test, y_test)
    policy_net = ImprovedResponseDQN(state_dim=15, action_dim=8).to(device)
    target_net = ImprovedResponseDQN(state_dim=15, action_dim=8).to(device)
    target_net.load_state_dict(policy_net.state_dict())

    optimizer   = optim.AdamW(policy_net.parameters(), lr=0.0003, weight_decay=1e-5)
    replay_buf  = PrioritizedReplayBuffer(30000)
    epsilon     = 1.0
    epsilon_min = 0.02
    epsilon_dec = 0.9992
    gamma       = 0.99

    episode_rewards, accuracies = [], []
    pq_correct = pq_total = 0

    for ep in range(num_episodes):
        state = env.reset()
        if random.random() < epsilon:
            action = random.randint(0, 7)
        else:
            with torch.no_grad():
                action = int(policy_net(state).argmax().item())

        next_state, reward, done, info = env.step(action)
        replay_buf.push(state, action, reward, next_state, done)

        if info['is_pq']:
            pq_total += 1
            if info['correct']: pq_correct += 1

        if len(replay_buf) > batch_size:
            s, a, r, ns, d = replay_buf.sample(batch_size)
            q  = policy_net(s).gather(1, a.unsqueeze(1)).squeeze()
            with torch.no_grad():
                na = policy_net(ns).argmax(dim=1)
                nq = target_net(ns).gather(1, na.unsqueeze(1)).squeeze()
                tq = r + gamma * nq * (1 - d)
            loss = F.smooth_l1_loss(q, tq)
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(policy_net.parameters(), 10.0)
            optimizer.step()

        episode_rewards.append(reward)
        accuracies.append(1.0 if info['correct'] else 0.0)
        epsilon = max(epsilon_min, epsilon * epsilon_dec)

        if (ep + 1) % 5 == 0:
            target_net.load_state_dict(policy_net.state_dict())

        if (ep + 1) % 500 == 0:
            avg_r  = np.mean(episode_rewards[-500:])
            avg_a  = np.mean(accuracies[-500:]) * 100
            pq_acc = (pq_correct / pq_total * 100) if pq_total > 0 else 0
            print(f"  Episode [{ep+1}/{num_episodes}] "
                  f"Reward: {avg_r:.1f} | Acc: {avg_a:.1f}% | "
                  f"PQ: {pq_acc:.1f}% | ε: {epsilon:.3f}")
            pq_correct = pq_total = 0

    return policy_net, episode_rewards, accuracies


# ============================================================================
# EVALUATION
# ============================================================================
def evaluate_system(detector, response_agent, threat_names,
                    X_test, y_test, num_tests=2000):
    print("\n📊 EVALUATING SYSTEM")
    env = BetterResponseEnvironment(detector, threat_names, X_test, y_test)

    y_true_det,  y_pred_det  = [], []
    y_true_resp, y_pred_resp = [], []
    total_reward = 0
    confidences  = []
    pq_true, pq_pred = [], []
    pq_resp_correct = pq_resp_total = 0

    for _ in range(num_tests):
        state  = env.reset()
        is_pq  = env.true_label >= 8
        y_true_det.append(env.true_label)
        y_pred_det.append(env.detected_threat)
        confidences.append(env.confidence)
        if is_pq:
            pq_true.append(env.true_label)
            pq_pred.append(env.detected_threat)
        with torch.no_grad():
            action = int(response_agent(state).argmax().item())
        optimal = env.optimal_actions[env.true_label]
        y_true_resp.append(optimal)
        y_pred_resp.append(action)
        _, reward, _, info = env.step(action)
        total_reward += reward
        if is_pq:
            pq_resp_total += 1
            if info['correct']: pq_resp_correct += 1

    det_acc  = accuracy_score(y_true_det,  y_pred_det)  * 100
    resp_acc = accuracy_score(y_true_resp, y_pred_resp) * 100
    p, r, f1, _ = precision_recall_fscore_support(
        y_true_det, y_pred_det, average='weighted', zero_division=0)
    pq_det_acc  = accuracy_score(pq_true, pq_pred) * 100 if pq_true else 0
    pq_resp_acc = (pq_resp_correct / pq_resp_total * 100) if pq_resp_total > 0 else 0

    print(f"   Detection   : {det_acc:.2f}%")
    print(f"   PQ Detection: {pq_det_acc:.2f}%")
    print(f"   Response    : {resp_acc:.2f}%")
    print(f"   PQ Response : {pq_resp_acc:.2f}%")
    print(f"   F1-Score    : {f1:.3f}")

    return {
        'detection_accuracy'   : det_acc,
        'response_accuracy'    : resp_acc,
        'pq_detection_accuracy': pq_det_acc,
        'pq_response_accuracy' : pq_resp_acc,
        'f1_score'             : f1,
        'precision'            : p, 'recall': r,
        'avg_confidence'       : np.mean(confidences) * 100,
        'avg_reward'           : total_reward / num_tests,
        'y_true_detection'     : y_true_det,
        'y_pred_detection'     : y_pred_det
    }


# ============================================================================
# VISUALIZATION  — ax1 now shows train loss + test loss
# ============================================================================
def create_visualizations(results, lora_history, episode_rewards,
                          response_accs, threat_names,
                          baseline_results, lora_params, total_params):
    print("\n📊 CREATING VISUALIZATIONS")
    fig = plt.figure(figsize=(22, 14))
    short_names = [n if n.startswith('PQ-') else n[:6] for n in threat_names]

    # ── 1. LoRA training curves  (train loss + test loss + accuracies) ────
    ax1      = plt.subplot(3, 4, 1)
    ax1_twin = ax1.twinx()

    # Loss lines on ax1
    ax1.plot(lora_history['train_losses'],
             color='#2196F3', linestyle='-',  linewidth=2,
             alpha=0.8, label='Train Loss')
    ax1.plot(lora_history['test_losses'],
             color='#FF5722', linestyle='--', linewidth=2,
             alpha=0.8, label='Test Loss')

    # Accuracy lines on twin axis
    ax1_twin.plot(lora_history['train_accs'],
                  color='#4CAF50', linestyle='-',  linewidth=2,
                  alpha=0.7, label='Train Acc')
    ax1_twin.plot(lora_history['test_accs'],
                  color='#FF9800', linestyle='--', linewidth=2,
                  alpha=0.7, label='Test Acc')

    ax1.set_xlabel('Epoch', fontweight='bold')
    ax1.set_ylabel('Loss',        fontweight='bold')
    ax1_twin.set_ylabel('Accuracy (%)', fontweight='bold')
    ax1.set_title('LoRA Fine-tuning Curves', fontweight='bold', fontsize=12)
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='upper left',  fontsize=8)
    ax1_twin.legend(loc='upper right', fontsize=8)

    # ── 2. Response agent rewards ─────────────────────────────────────────
    ax2 = plt.subplot(3, 4, 2)
    w   = 100
    smoothed = np.convolve(episode_rewards, np.ones(w)/w, mode='valid')
    ax2.plot(smoothed, color='#4CAF50', linewidth=2)
    ax2.fill_between(range(len(smoothed)), smoothed, alpha=0.3, color='green')
    ax2.axhline(y=0, color='r', linestyle='--', alpha=0.5, linewidth=2)
    ax2.set_xlabel('Episode', fontweight='bold')
    ax2.set_ylabel('Avg Reward', fontweight='bold')
    ax2.set_title('Response Agent Learning', fontweight='bold', fontsize=12)
    ax2.grid(True, alpha=0.3)

    # ── 3. Response accuracy ──────────────────────────────────────────────
    ax3 = plt.subplot(3, 4, 3)
    smoothed_acc = np.convolve(response_accs, np.ones(w)/w, mode='valid') * 100
    ax3.plot(smoothed_acc, color='purple', linewidth=2)
    ax3.fill_between(range(len(smoothed_acc)), smoothed_acc, alpha=0.3, color='purple')
    ax3.set_ylim([0, 100])
    ax3.set_xlabel('Episode', fontweight='bold')
    ax3.set_ylabel('Accuracy (%)', fontweight='bold')
    ax3.set_title('Response Accuracy', fontweight='bold', fontsize=12)
    ax3.grid(True, alpha=0.3)

    # ── 4. Confusion matrix ───────────────────────────────────────────────
    ax4 = plt.subplot(3, 4, 4)
    cm  = confusion_matrix(results['y_true_detection'], results['y_pred_detection'])
    cm_n = cm.astype('float') / np.maximum(cm.sum(axis=1)[:,None], 1) * 100
    cm_n = np.nan_to_num(cm_n)
    sns.heatmap(cm_n, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax4,
                xticklabels=short_names, yticklabels=short_names,
                linewidths=0.5, linecolor='gray',
                vmin=0, vmax=100, square=True, annot_kws={'size': 7})
    ax4.set_xlabel('Predicted', fontweight='bold')
    ax4.set_ylabel('True',      fontweight='bold')
    ax4.set_title('Confusion Matrix', fontweight='bold', fontsize=12)
    plt.setp(ax4.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=8)
    plt.setp(ax4.yaxis.get_majorticklabels(), rotation=0,  fontsize=8)

    # ── 5. Baseline vs LoRA grouped bar ───────────────────────────────────
    ax5 = plt.subplot(3, 4, 5)
    labels_   = ['Detection', 'PQ Detect', 'Response', 'F1×100']
    base_vals = [
        baseline_results['detection_accuracy'],
        baseline_results['pq_detection_accuracy'],
        baseline_results['response_accuracy'],
        baseline_results['f1_score'] * 100
    ]
    lora_vals = [
        results['detection_accuracy'],
        results['pq_detection_accuracy'],
        results['response_accuracy'],
        results['f1_score'] * 100
    ]
    x_  = np.arange(len(labels_))
    bw  = 0.35
    ax5.bar(x_ - bw/2, base_vals, bw, label='Baseline',
            color='#4299e1', edgecolor='black', linewidth=1)
    ax5.bar(x_ + bw/2, lora_vals, bw, label='LoRA',
            color='#48bb78', edgecolor='black', linewidth=1)
    ax5.set_xticks(x_); ax5.set_xticklabels(labels_, fontsize=9)
    ax5.set_ylim([0, 105])
    ax5.set_ylabel('Score (%)', fontweight='bold')
    ax5.set_title('Baseline vs LoRA', fontweight='bold', fontsize=12)
    ax5.legend(fontsize=9); ax5.grid(True, axis='y', alpha=0.3)
    for xi, (bv, lv) in enumerate(zip(base_vals, lora_vals)):
        ax5.text(xi - bw/2, bv + 0.5, f'{bv:.1f}', ha='center', fontsize=7)
        ax5.text(xi + bw/2, lv + 0.5, f'{lv:.1f}', ha='center', fontsize=7)

    # ── 6. Threat distribution ────────────────────────────────────────────
    ax6 = plt.subplot(3, 4, 6)
    unique, counts = np.unique(results['y_true_detection'], return_counts=True)
    colors_ = ['#2E7D32' if i < 8 else '#C62828' for i in unique]
    ax6.bar([short_names[i] for i in unique], counts,
            color=colors_, alpha=0.7, edgecolor='black', linewidth=1)
    ax6.set_ylabel('Count', fontweight='bold')
    ax6.set_title('Threat Distribution', fontweight='bold', fontsize=12)
    ax6.grid(True, axis='y', alpha=0.3)
    plt.setp(ax6.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=8)

    # ── 7. Parameter efficiency pie ───────────────────────────────────────
    ax7 = plt.subplot(3, 4, 7)
    frozen = total_params - lora_params
    eff    = (1 - lora_params / total_params) * 100
    wedges, texts, autotexts = ax7.pie(
        [frozen, lora_params],
        labels=['Frozen Base', 'LoRA Adapters'],
        autopct='%1.1f%%',
        colors=['#4A5568', '#48BB78'],
        explode=(0.05, 0.1)
    )
    ax7.set_title(f'Parameter Efficiency: {eff:.1f}%',
                  fontweight='bold', fontsize=12)

    # ── 8. Key metrics box ────────────────────────────────────────────────
    ax8 = plt.subplot(3, 4, 8)
    improvement = results['detection_accuracy'] - baseline_results['detection_accuracy']
    ax8.text(0.5, 0.80, f"LoRA Accuracy\n{results['detection_accuracy']:.2f}%",
             ha='center', va='center', fontsize=22, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.5',
                       facecolor='lightblue', edgecolor='navy', linewidth=2))
    ax8.text(0.5, 0.45,
             f"vs Baseline: {baseline_results['detection_accuracy']:.2f}%\n"
             f"Improvement: {improvement:+.2f}%",
             ha='center', va='center', fontsize=13, fontweight='bold',
             color='darkgreen' if improvement >= 0 else 'darkred')
    ax8.text(0.5, 0.15,
             f"F1: {results['f1_score']:.3f}  |  Conf: {results['avg_confidence']:.1f}%",
             ha='center', va='center', fontsize=11, style='italic')
    ax8.axis('off')
    ax8.set_title('Key Metrics', fontweight='bold', fontsize=12)

    # ── 9. PQ performance box ─────────────────────────────────────────────
    ax9 = plt.subplot(3, 4, 9)
    pq_imp = results['pq_detection_accuracy'] - baseline_results['pq_detection_accuracy']
    ax9.text(0.5, 0.80, f"PQ Detection\n{results['pq_detection_accuracy']:.2f}%",
             ha='center', va='center', fontsize=22, fontweight='bold', color='darkred',
             bbox=dict(boxstyle='round,pad=0.5',
                       facecolor='lightcoral', edgecolor='darkred', linewidth=2))
    ax9.text(0.5, 0.45,
             f"vs Baseline: {baseline_results['pq_detection_accuracy']:.2f}%\n"
             f"Improvement: {pq_imp:+.2f}%",
             ha='center', va='center', fontsize=13, fontweight='bold',
             color='darkgreen' if pq_imp >= 0 else 'darkred')
    ax9.text(0.5, 0.15, "🔐 Post-Quantum Ready",
             ha='center', va='center', fontsize=12, style='italic')
    ax9.axis('off')
    ax9.set_title('Post-Quantum Performance', fontweight='bold', fontsize=12)

    # ── 10. Response performance box ─────────────────────────────────────
    ax10 = plt.subplot(3, 4, 10)
    resp_imp = results['response_accuracy'] - baseline_results['response_accuracy']
    ax10.text(0.5, 0.80, f"Response Acc\n{results['response_accuracy']:.2f}%",
              ha='center', va='center', fontsize=22, fontweight='bold',
              bbox=dict(boxstyle='round,pad=0.5',
                        facecolor='lightgreen', edgecolor='darkgreen', linewidth=2))
    ax10.text(0.5, 0.45,
              f"vs Baseline: {baseline_results['response_accuracy']:.2f}%\n"
              f"Improvement: {resp_imp:+.2f}%",
              ha='center', va='center', fontsize=13, fontweight='bold',
              color='darkgreen' if resp_imp >= 0 else 'darkred')
    ax10.text(0.5, 0.15, "🤖 Adaptive Learning Active",
              ha='center', va='center', fontsize=12, style='italic')
    ax10.axis('off')
    ax10.set_title('Response Performance', fontweight='bold', fontsize=12)

    # ── 11. Adapter size vs full model ────────────────────────────────────
    ax11 = plt.subplot(3, 4, 11)
    full_mb    = total_params * 4 / (1024 * 1024)
    adapter_mb = lora_params  * 4 / (1024 * 1024)
    bars = ax11.bar(
        ['Full Model\n(detector.pth)', 'LoRA Adapter\n(lora_adapter.pth)'],
        [full_mb, adapter_mb],
        color=['#4299e1', '#48bb78'], edgecolor='black', linewidth=1.5
    )
    for bar in bars:
        h = bar.get_height()
        ax11.text(bar.get_x() + bar.get_width()/2., h + 0.01,
                  f'{h:.3f} MB', ha='center', va='bottom',
                  fontsize=10, fontweight='bold')
    ax11.set_ylabel('Size (MB)', fontweight='bold')
    ax11.set_title(f'Adapter is {full_mb/adapter_mb:.0f}x Smaller',
                   fontweight='bold', fontsize=12)
    ax11.grid(True, axis='y', alpha=0.3)

    # ── 12. Summary box ───────────────────────────────────────────────────
    ax12 = plt.subplot(3, 4, 12)
    ax12.axis('off')
    benefits = [
        f"✓ Loaded from detector.pth (pretrained)",
        f"✓ Same scaler.pkl as baseline — correct norm",
        f"✓ Encoder FROZEN — low-level features preserved",
        f"✓ deep_features + classifier TRAINABLE",
        f"✓ LoRA adapters on all linear layers",
        f"✓ {lora_params:,} / {total_params:,} LoRA params",
        f"✓ {eff:.1f}% parameter efficiency",
        f"✓ Adapter: {adapter_mb:.3f} MB  vs  Full: {full_mb:.2f} MB",
        f"✓ Detection  : {improvement:+.2f}% vs baseline",
        f"✓ PQ Detect  : {pq_imp:+.2f}% vs baseline",
    ]
    ax12.text(0.5, 0.97, "LORA FINE-TUNING RESULTS",
              ha='center', va='top', fontsize=12,
              fontweight='bold', color='#48BB78')
    for i, b in enumerate(benefits):
        ax12.text(0.04, 0.87 - i * 0.095, b, ha='left', va='top', fontsize=9,
                  bbox=dict(boxstyle='round,pad=0.3',
                            facecolor='#F0FFF4', edgecolor='#68D391', linewidth=1))
    ax12.set_xlim(0, 1); ax12.set_ylim(0, 1)

    plt.suptitle(
        'Digital Hygiene — Proper LoRA Fine-tuning (Loaded from detector.pth)',
        fontsize=15, fontweight='bold', y=0.98)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    return fig


In [2]:

# ============================================================================
# MAIN
# ============================================================================
def main():
    print("\n" + "=" * 70)
    print("🚀 PROPER LORA FINE-TUNING PIPELINE")
    print(f"   Base model : {BASELINE_DETECTOR_PATH}")
    print(f"   Scaler     : {BASELINE_SCALER_PATH}  ← same as baseline")
    print(f"   Epochs     : {LORA_FINETUNE_EPOCHS}")
    print(f"   LR         : {LORA_LR}")
    print("=" * 70)

    start_time = time.time()

    # ── Load baseline results ─────────────────────────────────────────────
    with open(BASELINE_RESULTS_PATH, 'r') as f:
        baseline_results = json.load(f)
    print(f"\n✓ Baseline — Detection: {baseline_results['detection_accuracy']:.2f}% | "
          f"PQ: {baseline_results['pq_detection_accuracy']:.2f}%")

    # ── Load baseline scaler  (KEY FIX) ──────────────────────────────────
    print(f"\n📥 Loading scaler from: {BASELINE_SCALER_PATH}")
    with open(BASELINE_SCALER_PATH, 'rb') as f:
        baseline_scaler = pickle.load(f)
    print("   ✓ Scaler loaded — same normalization as detector.pth")

    # ── Load dataset with baseline scaler ────────────────────────────────
    print("\n[1/5] Loading Dataset...")
    X_train      = np.load("/home/user2/archive(9)/X_train.npy")
    X_test       = np.load("/home/user2/archive(9)/X_test.npy")
    y_train      = np.load("/home/user2/archive(9)/y_train.npy")
    y_test       = np.load("/home/user2/archive(9)/y_test.npy")
    num_features = X_train.shape[1]
    print(f"   ✓ X_train: {X_train.shape} | X_test: {X_test.shape}")
    loader = ImprovedDataLoader(scaler=baseline_scaler)
    threat_names = loader.threat_names
    

    # ── Build LoRA model and load pretrained weights ──────────────────────
    print("\n[2/5] Building LoRA Model and Loading detector.pth...")
    lora_model = LoRAThreatDetector(
        input_dim=num_features, num_classes=12, lora_config=LORA_CONFIG
    ).to(device)

    lora_model = load_baseline_into_lora(lora_model, BASELINE_DETECTOR_PATH)
    lora_params, total_params = lora_model.count_lora_parameters()

    print(f"\n📊 LoRA Parameters : {lora_params:,}")
    print(f"   Total Parameters: {total_params:,}")
    print(f"   Adapter size    : {lora_params * 4 / (1024*1024):.3f} MB")

    # ── Fine-tune ─────────────────────────────────────────────────────────
    print("\n[3/5] Fine-tuning LoRA Adapters + Classifier Head...")
    lora_history = train_lora_only(
        lora_model, X_train, y_train, X_test, y_test,
        num_epochs=LORA_FINETUNE_EPOCHS, batch_size=BATCH_SIZE
    )

    # ── Save ─────────────────────────────────────────────────────────────
    output_path = Path(OUTPUT_FOLDER)
    output_path.mkdir(parents=True, exist_ok=True)

    adapter_mb = save_lora_adapter(lora_model, output_path / "lora_adapter.pth")

    # ── Train response agent ──────────────────────────────────────────────
    print("\n[4/5] Training Response Agent with LoRA Detector...")
    response_agent, episode_rewards, response_accs = train_response_agent(
        lora_model, loader.threat_names, X_test, y_test,
        num_episodes=RESPONSE_EPISODES, batch_size=BATCH_SIZE
    )
    torch.save(response_agent.state_dict(), output_path / "response_agent_lora.pth")
    print("💾 response_agent_lora.pth saved")

    # ── Evaluate ──────────────────────────────────────────────────────────
    print("\n[5/5] Evaluating System...")
    results = evaluate_system(
        lora_model, response_agent, loader.threat_names, X_test, y_test
    )

    # ── Visualize ─────────────────────────────────────────────────────────
    fig = create_visualizations(
        results, lora_history, episode_rewards, response_accs,
        loader.threat_names, baseline_results, lora_params, total_params
    )
    fig.savefig(output_path / "lora_results.png", dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"💾 lora_results.png saved")

    # ── Save LoRA summary ─────────────────────────────────────────────────
    lora_summary = {
        **{k: float(v) if isinstance(v, (np.floating, float)) else v
           for k, v in results.items()
           if k not in ('y_true_detection', 'y_pred_detection')},
        'lora_config'          : LORA_CONFIG,
        'lora_params'          : int(lora_params),
        'total_params'         : int(total_params),
        'param_efficiency'     : float((1 - lora_params/total_params)*100),
        'adapter_size_mb'      : float(adapter_mb),
        'base_accuracy'        : float(baseline_results['detection_accuracy']),
        'lora_accuracy'        : float(results['detection_accuracy']),
        'improvement'          : float(results['detection_accuracy'] -
                                       baseline_results['detection_accuracy']),
        'training_time_seconds': float(time.time() - start_time),
        'timestamp'            : datetime.now().isoformat()
    }
    with open(output_path / "lora_results_summary.json", 'w') as f:
        json.dump(lora_summary, f, indent=2)
    print("💾 lora_results_summary.json saved")

    # ── Final summary ─────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("✅ LORA FINE-TUNING COMPLETE")
    print("=" * 70)
    imp    = results['detection_accuracy']    - baseline_results['detection_accuracy']
    pq_imp = results['pq_detection_accuracy'] - baseline_results['pq_detection_accuracy']
    print(f"\n   Baseline Detection : {baseline_results['detection_accuracy']:.2f}%")
    print(f"   LoRA Detection     : {results['detection_accuracy']:.2f}%  ({imp:+.2f}%)")
    print(f"\n   Baseline PQ        : {baseline_results['pq_detection_accuracy']:.2f}%")
    print(f"   LoRA PQ            : {results['pq_detection_accuracy']:.2f}%  ({pq_imp:+.2f}%)")
    print(f"\n   LoRA params        : {lora_params:,} / {total_params:,}")
    print(f"   Adapter size       : {adapter_mb:.3f} MB")
    print(f"   Training time      : {time.time()-start_time:.1f}s")
    print("=" * 70)

    return lora_model, response_agent, results, lora_summary


if __name__ == "__main__":
    lora_model, response_agent, results, lora_summary = main()


🚀 PROPER LORA FINE-TUNING PIPELINE
   Base model : /home/user2/archive(9)/detector.pth
   Scaler     : /home/user2/archive(9)/scaler.pkl  ← same as baseline
   Epochs     : 100
   LR         : 0.001

✓ Baseline — Detection: 89.20% | PQ: 82.93%

📥 Loading scaler from: /home/user2/archive(9)/scaler.pkl
   ✓ Scaler loaded — same normalization as detector.pth

[1/5] Loading Dataset...
   ✓ X_train: (111993, 39) | X_test: (27999, 39)

[2/5] Building LoRA Model and Loading detector.pth...

📥 Loading baseline weights from: /home/user2/archive(9)/detector.pth
   ✓ Loaded      : 44 weight tensors
   ✓ LoRA keys   : 12 (initialised fresh — correct)

🔒 FREEZE STRATEGY
   Frozen params    : 87,296
   Trainable params : 98,674
   LoRA params      : 14,744 / 185,970
   Param efficiency : 92.1%

📊 LoRA Parameters : 14,744
   Total Parameters: 185,970
   Adapter size    : 0.056 MB

[3/5] Fine-tuning LoRA Adapters + Classifier Head...

LORA FINE-TUNING
  Epochs     : 100
  LR         : 0.001
  Batch s

LoRA Fine-tuning:  10%|█         | 10/100 [00:53<08:01,  5.35s/it]

  Epoch [10/100] Train Loss: 0.0058 | Test Loss: 0.0112 | Train: 89.67% | Val: 87.24% | Test: 87.27%


LoRA Fine-tuning:  20%|██        | 20/100 [01:48<07:16,  5.46s/it]

  Epoch [20/100] Train Loss: 0.0053 | Test Loss: 0.0067 | Train: 90.33% | Val: 89.80% | Test: 90.10%


...

LoRA Fine-tuning:  90%|█████████ | 90/100 [08:09<00:54,  5.42s/it]

  Epoch [90/100] Train Loss: 0.0048 | Test Loss: 0.0113 | Train: 91.31% | Val: 87.60% | Test: 87.47%


LoRA Fine-tuning: 100%|██████████| 100/100 [09:03<00:00,  5.44s/it]

  Epoch [100/100] Train Loss: 0.0048 | Test Loss: 0.0067 | Train: 91.23% | Val: 90.79% | Test: 90.68%

✅ Best model restored (Val: 91.34%)

💾 LoRA adapter saved: /home/user2/archive(9)/lora_adapter.pth  (0.061 MB)

[4/5] Training Response Agent with LoRA Detector...

🤖 TRAINING RESPONSE AGENT  (episodes: 8000)


  Episode [500/8000] Reward: -17.9 | Acc: 16.6% | PQ: 20.3% | ε: 0.670
  Episode [1000/8000] Reward: -6.9 | Acc: 25.6% | PQ: 30.8% | ε: 0.449
  Episode [1500/8000] Reward: -7.0 | Acc: 24.0% | PQ: 35.4% | ε: 0.301
  Episode [2000/8000] Reward: 1.4 | Acc: 33.8% | PQ: 43.4% | ε: 0.202
  Episode [2500/8000] Reward: 10.7 | Acc: 44.2% | PQ: 52.2% | ε: 0.135
  Episode [3000/8000] Reward: 13.1 | Acc: 46.4% | PQ: 57.0% | ε: 0.091
  Episode [3500/8000] Reward: 20.4 | Acc: 54.8% | PQ: 65.4% | ε: 0.061
  Episode [4000/8000] Reward: 27.0 | Acc: 61.8% | PQ: 68.3% | ε: 0.041
  Episode [4500/8000] Reward: 25.6 | Acc: 60.6% | PQ: 66.7% | ε: 0.027
  Episode [5000/8000] Reward: 31.2 | Acc: 68.8% | PQ: 74.5% | ε: 0.020
  Episode [5500/8000] Reward: 37.5 | Acc: 72.6% | PQ: 84.0% | ε: 0.020
  Episode [6000/8000] Reward: 41.9 | Acc: 77.8% | PQ: 87.9% | ε: 0.020
  Episode [6500/8000] Reward: 43.0 | Acc: 78.2% | PQ: 98.0% | ε: 0.020
  Episode [7000/8000] Reward: 45.8 | Acc: 81.8% | PQ: 97.6% | ε: 0.020
  Episo